# Лабораторная работа № 1. Закупка упаковки

Вариант 11 · Google Colab · **Среда выполнения → Выполнить всё**.

In [ ]:
%pip install -q numpy==2.3.5 scipy==1.17.0 matplotlib==3.10.8

## Исходные данные

Спрос, лимиты поставщиков и тарифы получены небольшим случайным разбросом от шаблонов. Лимиты по тарифам заданы шаблоном. Используем отдельный `data.json`. Если файла нет, создаём его с `seed=10012`.

In [ ]:
import csv
import json
import os
from contextlib import redirect_stderr, redirect_stdout
from ctypes import CDLL
from pathlib import Path

import numpy as np
from IPython.display import JSON, Markdown, display
from scipy.optimize import linprog

SEED = 10012
TOL = 1e-7
STATUS = {0: "optimal", 1: "limit", 2: "infeasible", 3: "unbounded", 4: "solver_error"}

LAB_DIR = Path.cwd()
if LAB_DIR.name != "lr1":
    LAB_DIR = LAB_DIR / "lr1"
DATA_PATH = LAB_DIR / "data.json"
RESULTS = LAB_DIR / "results"
STAR_RESULTS = RESULTS / "star"


def packaging(rng):
    # Цена задаётся в руб./шт., объём — в тыс. шт.
    contract = np.array([[34, 52, 18, 28], [37, 50, 23, 31], [39, 57, 17, 33],
                         [42, 53, 21, 27], [40, 55, 24, 29]])
    premium = np.array([[10, 11, 9, 8], [9, 9, 9, 11], [8, 7, 10, 8],
                        [9, 9, 9, 9], [10, 8, 8, 9]])
    return {
        "seed": SEED, "suppliers": ["S1", "S2", "S3", "S4", "S5"],
        "types": ["Коробка", "Лоток", "Пакет", "Вкладыш"],
        "demand": ([85, 75, 90, 65] + rng.integers(-3, 4, 4)).tolist(),
        "capacity": ([85, 73, 68, 65, 70] + rng.integers(-3, 4, 5)).tolist(),
        "contract_limit": [[60, 28, 20, 14], [34, 53, 12, 17], [24, 18, 63, 12],
                           [16, 27, 24, 47], [23, 31, 20, 37]],
        "regular_limit": [[35, 32, 27, 27], [35, 29, 22, 24], [33, 32, 32, 31],
                          [31, 33, 33, 32], [36, 34, 34, 33]],
        "contract_price": (contract + rng.integers(0, 2, contract.shape)).tolist(),
        "regular_price": (contract + premium + rng.integers(0, 2, contract.shape)).tolist(),
    }


if not DATA_PATH.is_file():
    LAB_DIR.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(np.random.SeedSequence([SEED, 1]))
    DATA_PATH.write_text(
        json.dumps(packaging(rng), ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
data = json.loads(DATA_PATH.read_text(encoding="utf-8"))

## Решатель и сохранение результатов

In [ ]:
def write_csv(folder, name, headers, rows):
    path = folder / f"{name}.csv"
    path.parent.mkdir(exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(headers)
        writer.writerows(rows)


def write_check(folder, name, result):
    path = folder / f"{name}.json"
    path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def solve_lp(cost, *, label, verbose=True, **kwargs):
    print(f"\n=== {label} ===", flush=True)
    result = linprog(
        cost, method="highs-ds", options={"presolve": True, "disp": verbose}, **kwargs
    )
    print(f"{STATUS.get(result.status, result.status)} | {result.message}", flush=True)
    return result


def require_status(result, *allowed):
    if result.status not in allowed:
        raise RuntimeError(result.message)


def run_with_log(run, data, folder):
    folder.mkdir(parents=True, exist_ok=True)
    with (folder / "solver.log").open("w", encoding="utf-8", buffering=1) as log:
        # HiGHS пишет лог напрямую в stdout, минуя Python.
        stdout_fd = os.dup(1)
        try:
            os.dup2(log.fileno(), 1)
            with redirect_stdout(log), redirect_stderr(log):
                return run(data)
        finally:
            CDLL(None).fflush(None)
            os.dup2(stdout_fd, 1)
            os.close(stdout_fd)


def show_table(path):
    with path.open(encoding="utf-8", newline="") as file:
        header, *rows = csv.reader(file)
    table = [header, ["---"] * len(header), *rows]
    display(Markdown("\n".join("| " + " | ".join(row) + " |" for row in table)))

## Построение модели

In [ ]:
def inputs(data):
    price = np.stack((data["contract_price"], data["regular_price"]), axis=-1)
    limit = np.stack((data["contract_limit"], data["regular_limit"]), axis=-1)
    return price, limit, np.array(data["capacity"]), np.array(data["demand"])


def build_model(price, limit, capacity, demand):
    ns, nt, nk = price.shape
    columns = np.arange(price.size).reshape(price.shape)
    supplier = np.zeros((ns, price.size))
    packaging = np.zeros((nt, price.size))
    supplier[np.broadcast_to(np.arange(ns)[:, None, None], price.shape), columns] = 1
    packaging[np.broadcast_to(np.arange(nt)[None, :, None], price.shape), columns] = 1
    return dict(A_ub=supplier, b_ub=capacity, A_eq=packaging, b_eq=demand,
                bounds=np.column_stack((np.zeros(price.size), limit.ravel())))


def shortage_capacities(capacity):
    deficit = capacity.copy().astype(float)
    deficit[0] *= 0.8
    stress = deficit.copy()
    stress[1] *= 0.5
    return deficit, stress

## Независимая проверка

In [ ]:
def max_violation(values):
    return float(np.max(values, initial=0))


def verify_objective(checks, expected, actual):
    checks["objective_recomputed"] = float(actual)
    checks["objective_error"] = float(abs(expected - actual))
    if any(value > TOL for key, value in checks.items() if key != "objective_recomputed"):
        raise ValueError(f"Independent check failed: {checks}")
    return checks


def verify(plan, price, limit, capacity, demand, objective):
    checks = {
        "demand_balance": max_violation(abs(plan.sum(axis=(0, 2)) - demand)),
        "supplier_capacity": max_violation(plan.sum(axis=(1, 2)) - capacity),
        "tier_upper": max_violation(plan - limit),
        "nonnegative": max_violation(-plan),
    }
    return verify_objective(checks, objective, np.sum(plan * price))

## Эксперименты

Базовый, дефицитный и стрессовый сценарии, затем увеличение лимита поставщика с наибольшей по модулю теневой ценой.

In [ ]:
def run_lab(data):
    price, limit, capacity, demand = inputs(data)
    deficit, stress = shortage_capacities(capacity)
    scenarios = [("base", capacity, 0), ("deficit", deficit, 0), ("stress", stress, 2)]
    summary, plans, checks, marginals = [], [], {}, []
    for name, available, expected in scenarios:
        result = solve_lp(price.ravel(), label=name, **build_model(price, limit, available, demand))
        require_status(result, expected)
        summary.append([name, *available, available.sum(), demand.sum(), STATUS[result.status],
                        result.fun, result.nit])
        if result.status == 2:
            assert available.sum() < demand.sum() - TOL
            checks[name] = {"required": int(demand.sum()), "capacity": float(available.sum()),
                            "certified_shortfall": float(demand.sum() - available.sum())}
            continue
        plan = result.x.reshape(price.shape)
        checks[name] = verify(plan, price, limit, available, demand, result.fun)
        for i, supplier in enumerate(data["suppliers"]):
            for j, package in enumerate(data["types"]):
                plans.append([name, supplier, package, *plan[i, j], plan[i, j].sum(),
                              np.dot(plan[i, j], price[i, j])])
        if name == "base":
            for index, supplier in enumerate(data["suppliers"]):
                marginals.append(["supplier", supplier, "", "", result.ineqlin.residual[index],
                                  result.ineqlin.marginals[index]])
            for index, package in enumerate(data["types"]):
                marginals.append(["demand", "", package, "", result.eqlin.residual[index],
                                  result.eqlin.marginals[index]])
            for (i, j, k), index in np.ndenumerate(np.arange(price.size).reshape(price.shape)):
                marginals.append(["tier_upper", data["suppliers"][i], data["types"][j], k + 1,
                                  result.upper.residual[index], result.upper.marginals[index]])
                marginals.append(["tier_lower", data["suppliers"][i], data["types"][j], k + 1,
                                  result.lower.residual[index], result.lower.marginals[index]])
            base = result

    increased = capacity.copy()
    best = int(np.argmin(base.ineqlin.marginals))
    increased[best] += 1
    test = solve_lp(price.ravel(), label="best supplier +1", **build_model(price, limit, increased, demand))
    require_status(test, 0)
    checks["best_supplier_plus_one"] = verify(
        test.x.reshape(price.shape), price, limit, increased, demand, test.fun
    )
    summary.append(["best_supplier_plus_one", *increased, increased.sum(), demand.sum(),
                    STATUS[test.status], test.fun, test.nit])
    write_csv(RESULTS, "scenarios",
              ["scenario", *data["suppliers"], "capacity", "demand", "status", "cost", "iterations"],
              summary)
    write_csv(RESULTS, "solution",
              ["scenario", "supplier", "type", "contract", "regular", "total", "cost"], plans)
    write_csv(RESULTS, "marginals", ["kind", "supplier", "type", "tier", "slack", "marginal"],
              marginals)
    write_check(RESULTS, "checks", checks)


run_with_log(run_lab, data, RESULTS)

## Результаты

Планы, проверки и лог сохранены в папке `results`.

In [ ]:
show_table(RESULTS / "scenarios.csv")
display(JSON(json.loads((RESULTS / "checks.json").read_text(encoding="utf-8"))))

## ★ Эластичный фильтр

Минимизируем суммарное увеличение лимитов, затем находим самый дешёвый план при этом увеличении.

In [ ]:
def build_filter(price, limit, capacity, demand):
    model = build_model(price, limit, capacity, demand)
    count = len(capacity)
    model["A_ub"] = np.column_stack((model["A_ub"], -np.eye(count)))
    model["A_eq"] = np.column_stack((model["A_eq"], np.zeros((len(demand), count))))
    model["bounds"] = [*map(tuple, model["bounds"]), *([(0, None)] * count)]
    return model


def run_star(data):
    price, limit, capacity, demand = inputs(data)
    _, stress = shortage_capacities(capacity)
    model = build_filter(price, limit, stress, demand)
    repair_cost = np.r_[np.zeros(price.size), np.ones(len(stress))]
    original_cost = np.r_[price.ravel(), np.zeros(len(stress))]
    repair = solve_lp(repair_cost, label="minimum total capacity increase", **model)
    require_status(repair, 0)
    lower_bound = demand.sum() - stress.sum()
    assert abs(repair.fun - lower_bound) <= TOL

    model["A_eq"] = np.vstack((model["A_eq"], repair_cost))
    model["b_eq"] = np.r_[model["b_eq"], repair.fun]
    selection = solve_lp(original_cost, label="least-cost plan at minimum increase", **model)
    require_status(selection, 0)
    delta = selection.x[price.size:]
    plan = selection.x[:price.size].reshape(price.shape)
    checks = verify(plan, price, limit, stress + delta, demand, selection.fun)
    assert abs(delta.sum() - lower_bound) <= TOL and min(delta) >= -TOL
    checks["minimum_increase"] = float(delta.sum())
    checks["analytical_lower_bound"] = float(lower_bound)

    restored = solve_lp(price.ravel(), label="original LP with repaired capacity",
                        **build_model(price, limit, stress + delta, demand))
    require_status(restored, 0)
    checks["restored_model"] = verify(
        restored.x.reshape(price.shape), price, limit, stress + delta, demand, restored.fun
    )
    assert abs(restored.fun - selection.fun) <= TOL

    write_check(STAR_RESULTS, "checks", checks)
    write_csv(STAR_RESULTS, "repair", ["supplier", "stress_capacity", "increase", "repaired_capacity", "used"],
              zip(data["suppliers"], stress, delta, stress + delta, plan.sum(axis=(1, 2))))
    write_csv(STAR_RESULTS, "solution", ["supplier", "type", "contract", "regular"],
              ([supplier, kind, *plan[i, j]] for i, supplier in enumerate(data["suppliers"])
               for j, kind in enumerate(data["types"])))


run_with_log(run_star, data, STAR_RESULTS)
show_table(STAR_RESULTS / "repair.csv")
display(JSON(json.loads((STAR_RESULTS / "checks.json").read_text(encoding="utf-8"))))